# SciPy

**Advanced Computational Physics** &middot; Master in Physics &middot; USC

*X. Cid Vidal* (original material, 2025). Adapted by *J. A. Hernando Morata*.

**SciPy** provides the algorithms of numerical physics on top of the
[NumPy](./numpy.ipynb) arrays: linear algebra, interpolation, optimization and fitting.

**Objectives**

* solve linear systems and eigenvalue problems;
* **interpolate** exact data and **smooth** noisy data, and know the difference;
* **fit** models to data, with parameter uncertainties and a $\chi^2$ check.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Submodules

**SciPy** builds on NumPy: compiled and tested algorithms, organized in submodules.

| Submodule | Contents |
|:--|:--|
| `scipy.linalg` | linear algebra (more than `np.linalg`) |
| `scipy.optimize` | minimization, root finding, curve fitting |
| `scipy.interpolate` | interpolation and splines |
| `scipy.stats` | probability distributions and statistics |
| `scipy.constants` | physical constants |
| `scipy.integrate` | integration and ODEs |
| `scipy.signal`, `scipy.fft` | signal processing, Fourier transforms |
| `scipy.sparse` | sparse matrices and their solvers |

Submodules are imported explicitly:

In [ ]:
from scipy import constants, interpolate, linalg, optimize, stats

In [ ]:
print('c   =', constants.c, 'm/s')
print('N_A =', constants.N_A, '1/mol')
print(constants.physical_constants['electron mass'])    # (value, unit, uncertainty)

In [ ]:
rng = np.random.default_rng(4)
sample = stats.norm(loc=1, scale=2).rvs(size=1000, random_state=rng)
print(f'mean = {sample.mean():.3f}, std = {sample.std(ddof=1):.3f}')

## 2. Linear algebra

`np.linalg` covers the basics; `scipy.linalg` has the same functions and more
(LU and Cholesky factorizations, matrix exponential, structured solvers), all built on
LAPACK.

### 2.1 Linear systems

Solve $A\mathbf{x} = \mathbf{b}$ with `solve`, never with `inv(A) @ b`: one
factorization is faster and more accurate than an inverse.

In [ ]:
rng = np.random.default_rng(5)
n = 200
M = rng.normal(size=(n, n))
A = M.T @ M + 1e-3 * np.eye(n)     # symmetric positive definite
b = rng.normal(size=n)

x_np = np.linalg.solve(A, b)
x_sp = linalg.solve(A, b, assume_a='pos')    # uses that A is positive definite
print(np.allclose(A @ x_np, b), np.allclose(A @ x_sp, b), np.allclose(x_np, x_sp))

* $M^T M$ is symmetric positive semi-definite, but can be almost singular.
* **Tikhonov regularization**: adding $\varepsilon I$ moves every eigenvalue up by
  $\varepsilon$.
* **Condition number** $\kappa = \lambda_{\max}/\lambda_{\min}$: how much a relative
  error in $\mathbf{b}$ can grow in $\mathbf{x}$.
* Rule: you lose about $\log_{10}\kappa$ significant digits; $\kappa \sim 10^{16}$ is
  singular in float64.

In [ ]:
print(f'kappa(M^T M)          = {np.linalg.cond(M.T @ M):.1e}')
print(f'kappa(M^T M + 1e-3 I) = {np.linalg.cond(A):.1e}')

Without regularization $\kappa \approx 10^8$: about 8 of the 16 digits of float64 are
lost. With $\varepsilon = 10^{-3}$, $\kappa \approx 10^6$.

### 2.2 Eigenvalues and eigenvectors

For symmetric (or Hermitian) matrices use `eigh`: real eigenvalues in ascending order,
and orthonormal eigenvectors **in the columns** of `v`.

In [ ]:
rng = np.random.default_rng(6)
H = rng.normal(size=(80, 80))
H = H + H.T                       # symmetric
w, v = linalg.eigh(H)
print(f'smallest eigenvalue: {w[0]:.4f}')
print('H v = w v:   ', np.allclose(H @ v[:, 0], w[0] * v[:, 0]))
print('orthonormal: ', np.allclose(v.T @ v, np.eye(80)))

The first eigenvector is `v[:, 0]`. `v[0]` is the first **row**: a frequent bug, and a
silent one. The sign of an eigenvector is arbitrary, and can change from one machine to
another.

## 3. Interpolation and smoothing

Two different problems:

* **interpolation**: the data are exact; find a curve that passes **through** every
  point;
* **smoothing**: the data are noisy; find a curve that passes **near** the points, and
  follows the trend, not the noise.

### 3.1 Interpolation

* `np.interp(x, xp, fp)`: linear interpolation;
* `interpolate.CubicSpline(xp, fp)`: piecewise cubic, continuous up to the second
  derivative;
* `interpolate.make_interp_spline(xp, fp, k)`: spline of any degree `k`.

`interpolate.interp1d` is legacy: you will find it in older code, but do not use it in
new code.

In [ ]:
x_knots = np.linspace(0, 10, 10)
y_knots = np.sin(x_knots)
spline = interpolate.CubicSpline(x_knots, y_knots)

print(f'sin(5.5)  = {np.sin(5.5):.4f}')
print(f'linear    = {np.interp(5.5, x_knots, y_knots):.4f}')
print(f'cubic     = {spline(5.5):.4f}')

In [ ]:
x_fine = np.linspace(0, 10, 400)
plt.figure(figsize=(6, 3))
plt.plot(x_fine, np.sin(x_fine), 'k:', label='sin(x)')
plt.plot(x_knots, y_knots, 'o', label='samples')
plt.plot(x_fine, np.interp(x_fine, x_knots, y_knots), '--', label='np.interp')
plt.plot(x_fine, spline(x_fine), '-', label='CubicSpline')
plt.legend(fontsize=8)
plt.show()

A spline is a function object: it can be differentiated and integrated.

In [ ]:
print(f'integral over [0, pi]: {spline.integrate(0, np.pi):.4f} (exact 2)')
print(f'derivative at 5:       {spline(5, 1):.4f} (exact {np.cos(5):.4f})')

Not exact: 10 knots over $[0, 10]$ is about one knot per radian, and the spline only
approximates $\sin x$ between them.

### 3.2 Noisy data: interpolate or smooth?

A **smoothing spline** balances closeness to the data against curvature.

* `interpolate.make_smoothing_spline(x, y)`: chooses the balance by itself
  (generalized cross-validation).
* `interpolate.UnivariateSpline(x, y, s=...)` (older): asks for a bound
  $\sum_i (y_i - f(x_i))^2 \le s$; with noise $\sigma$, try $s \approx N\sigma^2$.

In [ ]:
rng = np.random.default_rng(2)
sigma_noise = 0.2
x_data = np.linspace(0, 10, 30)
y_data = np.sin(x_data) + sigma_noise * rng.standard_normal(x_data.size)

interpolated = interpolate.CubicSpline(x_data, y_data)
smoothed = interpolate.make_smoothing_spline(x_data, y_data)

In [ ]:
plt.figure(figsize=(6, 3))
plt.plot(x_fine, np.sin(x_fine), 'k:', label='sin(x)')
plt.plot(x_data, y_data, 'o', label='data')
plt.plot(x_fine, interpolated(x_fine), '--', label='interpolation')
plt.plot(x_fine, smoothed(x_fine), '-', label='smoothing spline')
plt.legend(fontsize=8)
plt.show()

In [ ]:
for name, f in [('interpolation', interpolated), ('smoothing', smoothed)]:
    rms = np.sqrt(np.mean((f(x_fine) - np.sin(x_fine))**2))
    print(f'{name:13s}: rms distance to sin(x) = {rms:.3f}')

* The interpolation reproduces the noise exactly; the smoothing spline is closer to the
  true curve.
* Interpolate exact values (tables, simulations); smooth or fit measurements.

<div class="admonition note">
<p class="admonition-title"><b>[!] Discuss — Interpolate or smooth?</b></p>

A colleague interpolates a calibration curve measured with 1% noise, to correct every
event of an experiment. What goes wrong? When would interpolation be the right choice?

</div>

<details>
<summary><b>Solution</b></summary>

* The interpolated curve carries the noise of each calibration point into every event: a
  correlated, systematic error, not averaged away by more events.
* Better: fit a physical model, or smooth. Interpolate only values that are exact, or much
  more precise than the use you make of them (tables, simulations, fine theory grids).

</details>

### 3.3 Scattered data in 2D

`interpolate.griddata(points, values, xi)` interpolates values known at irregular points
onto any set of points, for instance a grid. Outside the convex hull of the data it
returns `nan`.

In [ ]:
rng = np.random.default_rng(8)
pts = rng.uniform(-1, 1, size=(300, 2))
vals = np.sin(3 * pts[:, 0]) * np.cos(2 * pts[:, 1])

gx, gy = np.meshgrid(np.linspace(-1, 1, 100), np.linspace(-1, 1, 80))
gz = interpolate.griddata(pts, vals, (gx, gy), method='cubic')

plt.figure(figsize=(6, 3))
plt.imshow(gz, extent=(-1, 1, -1, 1), origin='lower', aspect='auto')
plt.colorbar()
plt.scatter(pts[:, 0], pts[:, 1], s=4, c='k')
plt.show()

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — Reconstruct a signal</b></p>

1. Build a signal on $[0, 1]$, 200 points, with sharp features, for instance
   $\sin(4\pi x)$ + a narrow peak at $x = 0.3$ + a step at $x = 0.7$, and Gaussian
   noise with $\sigma = 0.1$.
2. Remove 40% of the samples at random.
3. Reconstruct the signal with: `np.interp`; `CubicSpline`; a smoothing spline
   (`UnivariateSpline` with a tuned `s`, or `make_smoothing_spline`); a fit to a
   Fourier series with 10 harmonics (`np.linalg.lstsq`).
4. Compare the RMS distance to the true signal, and plot. Which method wins where?

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
def signal(x):
    peak = 0.8 * np.exp(-0.5 * ((x - 0.3) / 0.02)**2)
    return np.sin(4 * np.pi * x) + peak + 0.5 * (x > 0.7)

rng = np.random.default_rng(9)
sig = 0.1
x_all = np.linspace(0, 1, 200)
y_all = signal(x_all) + sig * rng.standard_normal(x_all.size)
keep = np.sort(rng.choice(x_all.size, size=120, replace=False))
xk, yk = x_all[keep], y_all[keep]

def fourier_basis(x, k_max=10):
    cols = [np.ones_like(x)]
    for k in range(1, k_max + 1):
        cols += [np.cos(2 * np.pi * k * x), np.sin(2 * np.pi * k * x)]
    return np.column_stack(cols)

coef, *_ = np.linalg.lstsq(fourier_basis(xk), yk, rcond=None)
xf = np.linspace(0, 1, 1000)
regions = {'peak': np.abs(xf - 0.3) < 0.06, 'step': np.abs(xf - 0.7) < 0.06,
           'edges': (xf < 0.05) | (xf > 0.95)}
regions['smooth'] = ~(regions['peak'] | regions['step'] | regions['edges'])
recos = {'linear': np.interp(xf, xk, yk),
         'cubic': interpolate.CubicSpline(xk, yk)(xf),
         'smoothing': interpolate.UnivariateSpline(xk, yk, s=xk.size * sig**2)(xf),
         'Fourier': fourier_basis(xf) @ coef}
plt.figure(figsize=(6, 3))
plt.plot(xk, yk, '.k', ms=3)
for name, yr in recos.items():
    d2 = (yr - signal(xf))**2
    rms = '  '.join(f'{k} {np.sqrt(d2[m].mean()):.3f}' for k, m in regions.items())
    print(f'{name:9s}: all {np.sqrt(d2.mean()):.3f}  {rms}')
    plt.plot(xf, yr, label=name)
plt.legend(fontsize=8)
plt.show()
```

* Smoothing spline: the smallest rms in every region (0.055 overall).
* Interpolations follow the noise; `CubicSpline` also overshoots at the edges (0.22).
* Fourier, 10 harmonics: too few to follow the narrow peak or the step; Gibbs
  oscillations at the step **and at the edges**, since the signal is not periodic.
* The best method depends on what you know about the signal.

</details>

## 4. Optimization and fitting

### 4.1 Minimization

`optimize.minimize(f, x0, method=...)` finds a **local** minimum of a scalar function of
several variables. The Rosenbrock function has its minimum at $(1, 1)$, at the bottom of
a narrow curved valley:

In [ ]:
def rosen(p):
    x, y = p
    return (1 - x)**2 + 100 * (y - x**2)**2

res = optimize.minimize(rosen, x0=[-1.2, 1.0], method='BFGS')
print(res.success, res.x.round(4))

Other methods: `Nelder-Mead` (no derivatives), `L-BFGS-B` (with bounds), and more. For
roots of equations: `optimize.root_scalar` and `optimize.root`.

### 4.2 Curve fitting: `curve_fit`

`optimize.curve_fit(f, x, y, p0, sigma)` fits a model `f(x, *params)` by minimizing

$$\chi^2 = \sum_i \left(\frac{y_i - f(x_i)}{\sigma_i}\right)^2,$$

and returns the best parameters and their **covariance matrix**.

* `absolute_sigma=True`: the $\sigma_i$ are true uncertainties; the covariance uses them
  as given.
* `absolute_sigma=False` (the default): the covariance is multiplied by the reduced
  $\chi^2$, $\chi^2/n_{\rm dof}$; only the relative $\sigma_i$ matter.

In [ ]:
def exp_model(t, amp, tau, offset):
    return amp * np.exp(-t / tau) + offset

rng = np.random.default_rng(10)
sigma_exp = 0.1
t_exp = np.linspace(0, 5, 50)
y_exp = exp_model(t_exp, 2.0, 1.3, 0.5) + sigma_exp * rng.standard_normal(t_exp.size)

In [ ]:
popt, pcov = optimize.curve_fit(exp_model, t_exp, y_exp, p0=[1.5, 1.0, 0.4],
                                sigma=np.full(t_exp.size, sigma_exp), absolute_sigma=True)
perr = np.sqrt(np.diag(pcov))
for name, val, err in zip(['amp', 'tau', 'offset'], popt, perr):
    print(f'{name:6s} = {val:.3f} ± {err:.3f}')

In [ ]:
plt.figure(figsize=(6, 3))
plt.errorbar(t_exp, y_exp, yerr=sigma_exp, fmt='o', ms=3, label='data')
plt.plot(t_exp, exp_model(t_exp, *popt), '-', label='fit')
plt.legend()
plt.show()

### 4.3 More control: `least_squares`

`optimize.least_squares(fun, x0)` minimizes $\frac{1}{2}\sum_i r_i^2$ of a vector of
residuals that **you** write.

* Accepts bounds and robust losses (`loss='soft_l1'`).
* Does not return a covariance. With residuals in units of $\sigma$,
  $r_i = (f(x_i) - y_i)/\sigma_i$, it is $(J^T J)^{-1}$, $J$ the Jacobian at the
  minimum.

In [ ]:
def sin_model(t, p):
    amp, omega, phi, offset = p
    return amp * np.sin(omega * t + phi) + offset

def sin_residuals(p, t, y, sigma):
    return (sin_model(t, p) - y) / sigma

rng = np.random.default_rng(11)
sigma_sin = 0.2
t_sin = np.linspace(0, 10, 300)
p_sin = [1.2, 2.1, 0.5, 0.1]
y_sin = sin_model(t_sin, p_sin) + sigma_sin * rng.standard_normal(t_sin.size)

In [ ]:
res = optimize.least_squares(sin_residuals, x0=[1.0, 2.0, 0.0, 0.0],
                             args=(t_sin, y_sin, sigma_sin))
cov = np.linalg.inv(res.jac.T @ res.jac)
for name, val, err, true in zip(['amp', 'omega', 'phi', 'offset'], res.x,
                                np.sqrt(np.diag(cov)), p_sin):
    print(f'{name:6s} = {val:6.3f} ± {err:.3f}   (true {true})')
print(f'chi2 / ndof = {2 * res.cost:.1f} / {t_sin.size - 4}')

In [ ]:
plt.figure(figsize=(6, 3))
plt.plot(t_sin, y_sin, '.', ms=3, label='data')
plt.plot(t_sin, sin_model(t_sin, res.x), '-', label='fit')
plt.legend()
plt.show()

`p0` matters: $(A, \varphi)$ and $(-A, \varphi + \pi)$ give the same curve, and $\chi^2$
has many local minima in $\omega$. Start near the true frequency.

### 4.4 Polynomial fits

* `np.polynomial.Polynomial.fit(x, y, deg)`: linear least-squares fit; returns a
  callable polynomial.
* It rescales $x$ internally for numerical stability: `.convert().coef` gives the
  coefficients in increasing powers of $x$.
* Old API, still common: `np.polyfit` + `np.poly1d` (highest power first).

In [ ]:
rng = np.random.default_rng(12)
x_poly = np.linspace(-1, 1, 50)
y_poly = 1.5 * x_poly**3 - 0.7 * x_poly + 0.2 + 0.1 * rng.standard_normal(x_poly.size)

p3 = np.polynomial.Polynomial.fit(x_poly, y_poly, deg=3)
print(p3.convert().coef.round(3))     # true: 0.2, -0.7, 0, 1.5

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Degree 15</b></p>

Fit the same 50 points with a polynomial of degree 15. **Before running**: which fit has
the smaller sum of squared residuals, degree 3 or 15? Which one would you trust to
predict $y$ at a new $x$?

</div>

In [ ]:
p15 = np.polynomial.Polynomial.fit(x_poly, y_poly, deg=15)
for name, p in [('deg 3', p3), ('deg 15', p15)]:
    print(f'{name}: sum of squared residuals = {np.sum((y_poly - p(x_poly))**2):.3f}')

x_line = np.linspace(-1, 1, 400)
plt.figure(figsize=(6, 3))
plt.plot(x_poly, y_poly, 'o', ms=3, label='data')
plt.plot(x_line, p3(x_line), label='deg 3')
plt.plot(x_line, p15(x_line), label='deg 15')
plt.legend()
plt.show()

<details>
<summary><b>Solution</b></summary>

Degree 15 always has the smaller residuals: it contains degree 3 as a particular case,
and has more parameters to follow the data. But it follows the **noise**: it wiggles
between the points and near the edges. This is **overfitting**: a smaller $\chi^2$ is not
a better model. Compare $\chi^2/n_{\rm dof}$, or test the fit on data not used to fit
it.

</details>

## 5. Example: two peaks in a spectrum

A common physics case: two Gaussian peaks over noise, with a known uncertainty $\sigma$
per point. Good **initial values** are essential with several peaks.

In [ ]:
def gauss(x, amp, mu, width):
    return amp * np.exp(-0.5 * ((x - mu) / width)**2)

def two_peaks(x, a1, mu1, s1, a2, mu2, s2):
    return gauss(x, a1, mu1, s1) + gauss(x, a2, mu2, s2)

rng = np.random.default_rng(16)
sigma_y = 0.05
x_spec = np.linspace(-5, 5, 300)
p_spec = [1.2, -1.0, 0.5, 0.8, 1.2, 0.8]
y_spec = two_peaks(x_spec, *p_spec) + sigma_y * rng.standard_normal(x_spec.size)

In [ ]:
p0 = [1.0, -0.8, 0.6, 0.7, 1.0, 1.0]
popt2, pcov2 = optimize.curve_fit(two_peaks, x_spec, y_spec, p0=p0,
                                  sigma=np.full(x_spec.size, sigma_y), absolute_sigma=True)
names = ['a1', 'mu1', 's1', 'a2', 'mu2', 's2']
for name, val, err, true in zip(names, popt2, np.sqrt(np.diag(pcov2)), p_spec):
    print(f'{name:3s} = {val:6.3f} ± {err:.3f}   (true {true})')

In [ ]:
plt.figure(figsize=(6, 3))
plt.plot(x_spec, y_spec, '.', ms=3, label='data')
plt.plot(x_spec, two_peaks(x_spec, *popt2), '-', label='fit')
plt.plot(x_spec, two_peaks(x_spec, *p_spec), '--', label='true')
plt.legend()
plt.show()

### 5.1 Residuals and goodness of fit

If the model is right and the $\sigma$ are correct:

* the **pulls** $(y_i - f(x_i))/\sigma$ are approximately $\mathcal{N}(0, 1)$ (only
  approximately: the parameters are fitted to the same data);
* $\chi^2$ follows a $\chi^2$ distribution with $n_{\rm dof} = N - n_{\rm par}$:
  $\chi^2/n_{\rm dof} \approx 1 \pm \sqrt{2/n_{\rm dof}}$;
* **p-value** $= P(\chi^2 \geq \chi^2_{\rm obs} \mid \text{model right})$: $p \approx 0.5$ is
  typical; $p \ll 0.05$ points to a wrong model or wrong $\sigma$.

Structure in the pulls means a wrong model.

In [ ]:
pulls = (y_spec - two_peaks(x_spec, *popt2)) / sigma_y
chi2 = np.sum(pulls**2)
ndof = x_spec.size - len(popt2)
print(f'chi2 / ndof = {chi2:.1f} / {ndof} = {chi2 / ndof:.2f}')
print(f'p-value = {stats.chi2.sf(chi2, ndof):.2f}')

In [ ]:
plt.figure(figsize=(6, 2))
plt.plot(x_spec, pulls, '.', ms=3)
plt.axhline(0, color='k', ls='--')
plt.ylabel('pull')
plt.show()

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — curve_fit or least_squares?</b></p>

1. Generate a spectrum with two Gaussian peaks over a linear background,
   $b_0 + b_1 x$, with known $\sigma$ per point.
2. Fit it with `curve_fit` (with `sigma` and `absolute_sigma=True`) and with
   `least_squares` (covariance from the Jacobian).
3. Compare the parameters and their uncertainties. Compute $\chi^2/n_{\rm dof}$ and the
   p-value. What changes if you forget `absolute_sigma=True`?

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
def peaks_bkg(x, a1, mu1, s1, a2, mu2, s2, b0, b1):
    return two_peaks(x, a1, mu1, s1, a2, mu2, s2) + b0 + b1 * x

rng = np.random.default_rng(14)
sig = 0.05
xb = np.linspace(-5, 5, 300)
p_true = [1.0, -1.5, 0.4, 0.6, 1.0, 0.7, 0.3, 0.02]
yb = peaks_bkg(xb, *p_true) + sig * rng.standard_normal(xb.size)
p_init = [0.8, -1.3, 0.5, 0.5, 1.2, 0.8, 0.2, 0.]

p_cf, c_cf = optimize.curve_fit(peaks_bkg, xb, yb, p0=p_init,
                                sigma=np.full(xb.size, sig), absolute_sigma=True)
r_ls = optimize.least_squares(lambda p: (peaks_bkg(xb, *p) - yb) / sig, p_init)
c_ls = np.linalg.inv(r_ls.jac.T @ r_ls.jac)

for i, name in enumerate(['a1', 'mu1', 's1', 'a2', 'mu2', 's2', 'b0', 'b1']):
    print(f'{name:3s} curve_fit {p_cf[i]:7.4f} ± {np.sqrt(c_cf[i, i]):.4f}'
          f'   least_squares {r_ls.x[i]:7.4f} ± {np.sqrt(c_ls[i, i]):.4f}')
chi2_b = 2 * r_ls.cost
ndof_b = xb.size - len(p_init)
print(f'chi2/ndof = {chi2_b / ndof_b:.2f}, p-value = {stats.chi2.sf(chi2_b, ndof_b):.2f}')
```

* Both minimize the same $\chi^2$ (`curve_fit`: Levenberg-Marquardt by default;
  `least_squares`: a trust-region method).
* Same parameters and uncertainties, up to the precision of the minimization.
* Without `absolute_sigma=True`, `curve_fit` multiplies the covariance by
  $\chi^2/n_{\rm dof}$: a small change here, since it is close to 1; a large one if the
  $\sigma$ are wrong or the model is bad.

</details>

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — Fit an image (challenge)</b></p>

Fit a tilted 2D Gaussian to a noisy image with `least_squares`.

1. Generate an image of shape `(50, 60)` (rows follow $y$, columns $x$): a 2D
   Gaussian with amplitude, center $(x_0, y_0)$, widths $(\sigma_u, \sigma_v)$ along
   axes rotated by an angle $\theta$, plus a constant background and Gaussian noise.
2. Write a **vectorized** residual function (no loops; `.ravel()` turns the image into a
   vector) of the 7 parameters.
3. Fit, report the parameters with their uncertainties, and show the image of the
   pulls.

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
def tilted_gauss(p, x, y):
    amp, x0, y0, su, sv, theta, bkg = p
    c, s = np.cos(theta), np.sin(theta)
    u = (x - x0) * c + (y - y0) * s
    v = -(x - x0) * s + (y - y0) * c
    return amp * np.exp(-0.5 * ((u / su)**2 + (v / sv)**2)) + bkg

rng = np.random.default_rng(15)
sig = 0.05
xi, yi = np.meshgrid(np.linspace(-3, 3, 60), np.linspace(-2.5, 2.5, 50))
p_img = [1.0, 0.4, -0.3, 1.0, 0.5, 0.6, 0.1]
image = tilted_gauss(p_img, xi, yi) + sig * rng.standard_normal(xi.shape)

def img_residuals(p):
    return ((tilted_gauss(p, xi, yi) - image) / sig).ravel()

lower = [-np.inf, -np.inf, -np.inf, 0., 0., 0., -np.inf]        # theta in [0, pi/2]
upper = [np.inf, np.inf, np.inf, np.inf, np.inf, np.pi / 2, np.inf]
r_img = optimize.least_squares(img_residuals, x0=[0.8, 0., 0., 1.2, 0.6, 0.3, 0.],
                               bounds=(lower, upper))
err_img = np.sqrt(np.diag(np.linalg.inv(r_img.jac.T @ r_img.jac)))
for name, val, err, true in zip(['amp', 'x0', 'y0', 'su', 'sv', 'theta', 'bkg'],
                                r_img.x, err_img, p_img):
    print(f'{name:5s} = {val:6.3f} ± {err:.3f}   (true {true})')

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(8, 3))
ax0.imshow(image, origin='lower'); ax0.set_title('image')
im = ax1.imshow(img_residuals(r_img.x).reshape(image.shape), origin='lower', cmap='RdBu')
ax1.set_title('pulls'); fig.colorbar(im, ax=ax1)
plt.show()
```

* The model is not unique: $\theta \to \theta + \pi$ gives the same image, and so do
  $(\sigma_u, \sigma_v, \theta)$ and $(\sigma_v, \sigma_u, \theta + \pi/2)$.
* Without bounds the fit converges to one or another depending on the initial values.
* Bounds on $\theta$ spanning at most $\pi/2$, here $[0, \pi/2]$, remove the ambiguity.

</details>

## 6. Summary

* SciPy adds the algorithms: `linalg`, `interpolate`, `optimize`, `stats`.
* Interpolate exact data; smooth or fit noisy data.
* A fit is not finished without uncertainties and a goodness-of-fit check.

**Hints.** Good initial values and sensible bounds make fits converge. Look at the
residuals, not only at the numbers.

<details>
<summary><b>[i] Cheat sheet — SciPy</b></summary>

| | |
|:--|:--|
| `np.linalg.solve`, `linalg.eigh` | linear systems, symmetric eigenproblem |
| `np.interp`, `CubicSpline`, `make_interp_spline` | interpolation |
| `make_smoothing_spline`, `UnivariateSpline` | smoothing splines |
| `griddata` | scattered 2D data |
| `optimize.minimize` | local minimum |
| `curve_fit(..., sigma, absolute_sigma=True)` | fit, with covariance |
| `least_squares` + $(J^TJ)^{-1}$ | fit with control, covariance |
| `Polynomial.fit` | polynomial fit (old API: `polyfit`) |
| `stats.chi2.sf(chi2, ndof)` | p-value |

**Traps**: `v[0]` is not an eigenvector; interpolating noise; `curve_fit` without
`absolute_sigma`; overfitting.

</details>

**Further reading**: [SciPy user guide](https://docs.scipy.org/doc/scipy/tutorial/),
[Python Data Science Handbook](https://jakevdp.github.io/PythonDataScienceHandbook/).

### Check yourself

1. `w, v = linalg.eigh(H)`. Where is the eigenvector of `w[2]`?
2. You have 20 noisy measurements. Interpolate or smooth? Why?
3. `curve_fit` gives $\chi^2/n_{\rm dof} = 4$. What can be wrong?

<details>
<summary><b>Solutions</b></summary>

1. In the column `v[:, 2]`.
2. Smooth (or fit a model): an interpolation passes through every point and reproduces
   the noise.
3. The model does not describe the data, or the $\sigma$ are underestimated. Look at the
   pulls: structure points to the model, a uniform spread to the $\sigma$.

</details>